In [1]:
!pip install findspark

# Introducción al Análisis de Datos de COVID-19 en Corea del Sur

---

## 1. Contexto del Proyecto
Este cuaderno de Jupyter ha sido estructurado con base en el conjunto de datos de investigación **[NeurIPS 2020] Data Science for COVID-19 (DS4C)**, el cual se encuentra disponible en la plataforma **Kaggle**.

Esta base de datos recopila información epidemiológica oficial sobre los casos de contagio de COVID-19 en **Corea del Sur**, una de las naciones que destacó a nivel internacional por su transparencia, recolección masiva de métricas y efectividad en el control de las curvas de transmisión del virus.

A través de esta práctica, se simulará el flujo de trabajo de un Ingeniero de Datos o Científico de Datos, procesando información masiva y estructurada para transformarla en conocimiento analítico accionable.

---

## 2. Descripción de las Fuentes de Datos

Para el desarrollo del análisis se emplean dos archivos en formato tabular (`.csv`) interconectados:

* **`Case.csv` (Casos Reportados):** Este archivo consolida los focos de infección detectados a nivel nacional. Contiene campos clave como la provincia, la ciudad, si el contagio se debió a un brote grupal o aislado, la causa u origen del evento (ej. centros religiosos, lugares de trabajo, flujo internacional) y el conteo neto de casos confirmados.
* **`PatientInfo.csv` (Información de Pacientes):** Este archivo provee una granularidad a nivel individual de los contagiados. Almacena características demográficas anonimizadas como el género, el rango de edad, el país de origen, el estado del paciente (aislado, recuperado, fallecido) y, de vital importancia epidemiológica, el rastreo de quién originó el contagio (*redes de transmisión*).

---

## 3. Alcance y Objetivos del Cuaderno

El desarrollo de este cuaderno está guiado por objetivos específicos enfocados tanto en la analítica de negocio como en la optimización de los recursos de cómputo:

### A. Análisis Geográfico e Impacto
* Identificar con precisión las ciudades y provincias más afectadas por el virus a partir del volumen total de casos confirmados, consolidando una vista relacional limpia.

### B. Calidad y Depuración de Datos
* Aplicar técnicas de control de calidad para garantizar la unicidad de los registros, identificando y eliminando posibles duplicados en el padrón de pacientes.
* Aislar y analizar las cadenas de transmisión directa mediante el filtrado de registros con trazabilidad de contagio completa.

### C. Segmentación Demográfica
* Refinar el conjunto de datos mediante filtros demográficos (enfoque de género) y realizar limpieza de esquemas mediante la remoción de variables temporales o irrelevantes para este caso de estudio.

### D. Ingeniería y Optimización de Almacenamiento
* Diseñar una estrategia de persistencia y almacenamiento eficiente utilizando el formato columnar **Parquet**.
* Controlar la topología del clúster mediante el reparticionamiento de datos y la estructuración física de directorios de salida organizados por provincias utilizando modos de sobreescritura seguros.

In [2]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("COVID-19_NeurIPS 2020").getOrCreate()

In [5]:
casos = spark.read.option("header", "true").option("inferSchema", "true").csv("data/Case.csv")
pacientes = spark.read.option("header", "true").option("inferSchema", "true").csv("data/PatientInfo.csv")

In [7]:
# Visualizacio de los datos

casos.printSchema()

root
 |--  case_id: integer (nullable = true)
 |-- province: string (nullable = true)
 |-- city: string (nullable = true)
 |-- group: boolean (nullable = true)
 |-- infection_case: string (nullable = true)
 |-- confirmed: integer (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)



In [10]:
casos = casos.withColumnRenamed(" case_id", "case_id")

casos.printSchema()

root
 |-- case_id: integer (nullable = true)
 |-- province: string (nullable = true)
 |-- city: string (nullable = true)
 |-- group: boolean (nullable = true)
 |-- infection_case: string (nullable = true)
 |-- confirmed: integer (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)



In [11]:
casos.show()

+-------+--------+---------------+-----+--------------------+---------+---------+----------+
|case_id|province|           city|group|      infection_case|confirmed| latitude| longitude|
+-------+--------+---------------+-----+--------------------+---------+---------+----------+
|1000001|   Seoul|     Yongsan-gu| true|       Itaewon Clubs|      139|37.538621|126.992652|
|1000002|   Seoul|      Gwanak-gu| true|             Richway|      119| 37.48208|126.901384|
|1000003|   Seoul|        Guro-gu| true| Guro-gu Call Center|       95|37.508163|126.884387|
|1000004|   Seoul|   Yangcheon-gu| true|Yangcheon Table T...|       43|37.546061|126.874209|
|1000005|   Seoul|      Dobong-gu| true|     Day Care Center|       43|37.679422|127.044374|
|1000006|   Seoul|        Guro-gu| true|Manmin Central Ch...|       41|37.481059|126.894343|
|1000007|   Seoul|from other city| true|SMR Newly Planted...|       36|        -|         -|
|1000008|   Seoul|  Dongdaemun-gu| true|       Dongan Church|       17

In [12]:
from pyspark.sql.functions import col,  desc

In [21]:
casos.filter((col("city") != "-") & (col("city") != "from other city")).orderBy(desc("confirmed")).select("province","city","confirmed").show()

+-----------------+------------+---------+
|         province|        city|confirmed|
+-----------------+------------+---------+
|            Daegu|      Nam-gu|     4511|
|            Daegu|Dalseong-gun|      196|
|            Seoul|  Yongsan-gu|      139|
|            Daegu|      Seo-gu|      124|
|            Seoul|   Gwanak-gu|      119|
| Gyeongsangbuk-do|Cheongdo-gun|      119|
|Chungcheongnam-do|  Cheonan-si|      103|
|            Daegu|Dalseong-gun|      101|
|            Seoul|     Guro-gu|       95|
| Gyeongsangbuk-do| Bonghwa-gun|       68|
|      Gyeonggi-do| Seongnam-si|       67|
|      Gyeonggi-do|  Bucheon-si|       67|
| Gyeongsangbuk-do|Gyeongsan-si|       66|
|      Gyeonggi-do|Uijeongbu-si|       50|
|            Seoul|Yangcheon-gu|       43|
|            Seoul|   Dobong-gu|       43|
|            Seoul|     Guro-gu|       41|
| Gyeongsangbuk-do|  Yechun-gun|       40|
|            Busan|  Dongnae-gu|       39|
|            Daegu|     Dong-gu|       39|
+----------

In [22]:
# parte 2

pacientes.printSchema()

root
 |-- patient_id: long (nullable = true)
 |-- sex: string (nullable = true)
 |-- age: string (nullable = true)
 |-- country: string (nullable = true)
 |-- province: string (nullable = true)
 |-- city: string (nullable = true)
 |-- infection_case: string (nullable = true)
 |-- infected_by: string (nullable = true)
 |-- contact_number: string (nullable = true)
 |-- symptom_onset_date: string (nullable = true)
 |-- confirmed_date: date (nullable = true)
 |-- released_date: date (nullable = true)
 |-- deceased_date: date (nullable = true)
 |-- state: string (nullable = true)



In [23]:
pacientes.show()

+----------+------+---+-------+--------+------------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+
|patient_id|   sex|age|country|province|        city|      infection_case|infected_by|contact_number|symptom_onset_date|confirmed_date|released_date|deceased_date|   state|
+----------+------+---+-------+--------+------------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+
|1000000001|  male|50s|  Korea|   Seoul|  Gangseo-gu|     overseas inflow|       NULL|            75|        2020-01-22|    2020-01-23|   2020-02-05|         NULL|released|
|1000000002|  male|30s|  Korea|   Seoul| Jungnang-gu|     overseas inflow|       NULL|            31|              NULL|    2020-01-30|   2020-03-02|         NULL|released|
|1000000003|  male|50s|  Korea|   Seoul|   Jongno-gu|contact with patient| 2002000001|            17|              NULL|    2020-01-30|

In [29]:
pacientes.select("patient_id").count()

5165

In [57]:
from pyspark.sql.functions import col

final_df = paciente_contagios.filter((col("sex") == "female") & (col("sex").isNotNull())).drop("released_date","deceased_date")

# Guardar el datasets particionado en dos por la columna provincia
final_df.coalesce(2).write.partitionBy("province").mode("overwrite").parquet("./data/salida")

In [32]:
pacientes = pacientes.drop_duplicates(["patient_id"])

In [34]:
from pyspark.sql.functions import count

In [38]:
pacientes.select(count("infected_by").alias("conteo")).show()

+------+
|conteo|
+------+
|  1346|
+------+



In [44]:
paciente_contagios = pacientes.na.drop(subset=["infected_by"])

In [45]:
paciente_contagios.count()

1346

In [46]:
paciente_contagios.show()

+----------+------+---+-------+--------+------------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+
|patient_id|   sex|age|country|province|        city|      infection_case|infected_by|contact_number|symptom_onset_date|confirmed_date|released_date|deceased_date|   state|
+----------+------+---+-------+--------+------------+--------------------+-----------+--------------+------------------+--------------+-------------+-------------+--------+
|1000000003|  male|50s|  Korea|   Seoul|   Jongno-gu|contact with patient| 2002000001|            17|              NULL|    2020-01-30|   2020-02-19|         NULL|released|
|1000000005|female|20s|  Korea|   Seoul| Seongbuk-gu|contact with patient| 1000000002|             2|              NULL|    2020-01-31|   2020-02-24|         NULL|released|
|1000000006|female|50s|  Korea|   Seoul|   Jongno-gu|contact with patient| 1000000003|            43|              NULL|    2020-01-31|

In [53]:
final_df = paciente_contagios.filter((col("sex") == "female") & (col("sex").isNotNull())).drop("released_date","deceased_date")

+----------+------+---+-------+--------+-------------+--------------------+-----------+--------------+------------------+--------------+--------+
|patient_id|   sex|age|country|province|         city|      infection_case|infected_by|contact_number|symptom_onset_date|confirmed_date|   state|
+----------+------+---+-------+--------+-------------+--------------------+-----------+--------------+------------------+--------------+--------+
|1000000005|female|20s|  Korea|   Seoul|  Seongbuk-gu|contact with patient| 1000000002|             2|              NULL|    2020-01-31|released|
|1000000006|female|50s|  Korea|   Seoul|    Jongno-gu|contact with patient| 1000000003|            43|              NULL|    2020-01-31|released|
|1000000010|female|60s|  Korea|   Seoul|  Seongbuk-gu|contact with patient| 1000000003|             6|              NULL|    2020-02-05|released|
|1000000014|female|60s|  Korea|   Seoul|    Jongno-gu|contact with patient| 1000000013|            27|        2020-02-06|   